# Iris Classification - Optimization in Neural Networks

Notebook ini disusun sesuai spesifikasi tugas:
- dataset Iris
- normalisasi fitur
- one-hot encoding label
- split train/test
- model feedforward `4 -> 8 -> 6 -> 3`
- perbandingan **Full Batch Gradient Descent** vs **Gauss-Newton**
- evaluasi loss, training accuracy, testing accuracy
- visualisasi konvergensi

## 1) Import library

Notebook mengutamakan pengambilan data dari **UCI Machine Learning Repository** (`ucimlrepo`, id=53).  
Kalau akses/paket tidak tersedia, notebook otomatis fallback ke `sklearn.load_iris()` supaya tetap bisa dijalankan.

In [ ]:
import copy
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import torch
import torch.nn as nn

torch.set_num_threads(1)
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print("PyTorch version:", torch.__version__)

## 2) Load dataset dari UCI Iris (id=53)

In [ ]:
def load_iris_from_uci_or_fallback():
    try:
        from ucimlrepo import fetch_ucirepo

        iris = fetch_ucirepo(id=53)
        X_df = iris.data.features.copy()

        y_raw = iris.data.targets.copy()
        if isinstance(y_raw, pd.Series):
            y_df = y_raw.to_frame(name="class")
        else:
            y_df = y_raw.copy()
            if y_df.shape[1] == 1 and "class" not in y_df.columns:
                y_df.columns = ["class"]

        source_note = "Loaded directly from UCI Machine Learning Repository using ucimlrepo (id=53)."
        return X_df, y_df, source_note

    except Exception as e:
        from sklearn.datasets import load_iris

        iris = load_iris(as_frame=True)
        X_df = iris.data.copy()
        target_labels = pd.Series(iris.target).map(dict(enumerate(iris.target_names)))
        y_df = pd.DataFrame({"class": target_labels})

        source_note = f"UCI fetch failed, fallback to sklearn.load_iris(). Reason: {e}"
        return X_df, y_df, source_note


X_df, y_df, source_note = load_iris_from_uci_or_fallback()

print(source_note)
print("Shape fitur :", X_df.shape)
print("Shape target:", y_df.shape)
display(pd.concat([X_df.head(), y_df.head()], axis=1))

## 3) Normalisasi fitur, one-hot encoding label, dan train-test split

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_df)

try:
    encoder = OneHotEncoder(sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(sparse=False)

y_encoded = encoder.fit_transform(y_df)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y_encoded,
    test_size=0.20,
    random_state=SEED,
    stratify=y_df.values.ravel()
)

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

print("Train feature shape :", X_train_tensor.shape)
print("Test feature shape  :", X_test_tensor.shape)
print("Train label shape   :", y_train_tensor.shape)
print("Test label shape    :", y_test_tensor.shape)

pd.DataFrame(
    y_train[:5],
    columns=[f"class_{name}" for name in encoder.categories_[0]]
)

## 4) Bangun model feedforward dengan pendekatan sequential

In [ ]:
def build_model(seed=SEED):
    torch.manual_seed(seed)
    model = nn.Sequential(
        nn.Linear(4, 8),
        nn.ReLU(),
        nn.Linear(8, 6),
        nn.ReLU(),
        nn.Linear(6, 3),
        nn.Softmax(dim=1)
    )
    return model

base_model = build_model()
base_model

## 5) Forward pass awal: hitung prediksi output training data

In [ ]:
with torch.no_grad():
    y_hat_train_initial = base_model(X_train_tensor)

print("5 prediksi pertama sebelum training:")
display(pd.DataFrame(
    y_hat_train_initial[:5].numpy(),
    columns=[f"p({name})" for name in encoder.categories_[0]]
))

## 6) Loss function, akurasi, dan utilitas parameter

In [ ]:
def categorical_cross_entropy(y_true, y_pred, eps=1e-8):
    y_pred = torch.clamp(y_pred, eps, 1 - eps)
    return -(y_true * torch.log(y_pred)).sum(dim=1).mean()

def accuracy_score_tensor(y_true, y_pred):
    true_class = torch.argmax(y_true, dim=1)
    pred_class = torch.argmax(y_pred, dim=1)
    return (true_class == pred_class).float().mean().item()

def flatten_model_params(model):
    return torch.cat([param.detach().reshape(-1) for param in model.parameters()]).clone().requires_grad_(True)

def unpack_theta(theta):
    idx = 0

    W1 = theta[idx:idx+32].reshape(8, 4); idx += 32
    b1 = theta[idx:idx+8]; idx += 8

    W2 = theta[idx:idx+48].reshape(6, 8); idx += 48
    b2 = theta[idx:idx+6]; idx += 6

    W3 = theta[idx:idx+18].reshape(3, 6); idx += 18
    b3 = theta[idx:idx+3]; idx += 3

    return W1, b1, W2, b2, W3, b3

def forward_from_theta(theta, X):
    W1, b1, W2, b2, W3, b3 = unpack_theta(theta)

    z1 = X @ W1.T + b1
    a1 = torch.relu(z1)

    z2 = a1 @ W2.T + b2
    a2 = torch.relu(z2)

    z3 = a2 @ W3.T + b3
    y_hat = torch.softmax(z3, dim=1)

    return y_hat

# Verifikasi bahwa parameter flatten menghasilkan output yang sama
theta0 = flatten_model_params(base_model)
with torch.no_grad():
    pred_sequential = base_model(X_train_tensor)
    pred_theta = forward_from_theta(theta0, X_train_tensor)

print("Konsisten antara nn.Sequential dan forward_from_theta:",
      torch.allclose(pred_sequential, pred_theta))
print("Total parameter:", theta0.numel())

## 7) Full Batch Gradient Descent

Sesuai instruksi:
- menggunakan **seluruh data train** dalam setiap epoch
- gradien dihitung dengan **automatic differentiation**
- update parameter dilakukan **manual**

In [ ]:
def train_full_batch_gradient_descent(
    model,
    X_train,
    y_train,
    X_test,
    y_test,
    lr=0.20,
    epochs=150
):
    history = {
        "epoch": [],
        "loss": [],
        "train_acc": [],
        "test_acc": []
    }

    params = [p for p in model.parameters()]

    for epoch in range(1, epochs + 1):
        y_hat = model(X_train)
        loss = categorical_cross_entropy(y_train, y_hat)

        for p in params:
            if p.grad is not None:
                p.grad.zero_()

        loss.backward()

        with torch.no_grad():
            for p in params:
                p -= lr * p.grad

            train_pred = model(X_train)
            test_pred = model(X_test)

        history["epoch"].append(epoch)
        history["loss"].append(float(categorical_cross_entropy(y_train, train_pred)))
        history["train_acc"].append(accuracy_score_tensor(y_train, train_pred))
        history["test_acc"].append(accuracy_score_tensor(y_test, test_pred))

    return history

## 8) Gauss-Newton Method

Catatan:
- residual digunakan sebagai `r = y - y_hat`
- Jacobian dihitung terhadap residual yang sudah di-*flatten*
- ditambahkan **damping** kecil pada `J^T J` agar lebih stabil secara numerik

In [ ]:
def train_gauss_newton(
    theta,
    X_train,
    y_train,
    X_test,
    y_test,
    epochs=15,
    damping=1e-1,
    step_scale=1.0
):
    history = {
        "epoch": [],
        "loss": [],
        "train_acc": [],
        "test_acc": []
    }

    for epoch in range(1, epochs + 1):

        pred = forward_from_theta(theta, X_train)
        residual = (y_train - pred).reshape(-1)

        def residual_fn(current_theta):
            return (y_train - forward_from_theta(current_theta, X_train)).reshape(-1)

        J = torch.autograd.functional.jacobian(
            residual_fn,
            theta,
            vectorize=True
        )  # shape: (N*C, P)

        JTJ = J.T @ J
        JTr = J.T @ residual.detach()
        I = torch.eye(JTJ.shape[0], dtype=JTJ.dtype)

        delta = -torch.linalg.solve(JTJ + damping * I, JTr)

        with torch.no_grad():
            theta = (theta + step_scale * delta).detach().requires_grad_(True)

            train_pred = forward_from_theta(theta, X_train)
            test_pred = forward_from_theta(theta, X_test)

        history["epoch"].append(epoch)
        history["loss"].append(float(categorical_cross_entropy(y_train, train_pred)))
        history["train_acc"].append(accuracy_score_tensor(y_train, train_pred))
        history["test_acc"].append(accuracy_score_tensor(y_test, test_pred))

    return history, theta

## 9) Jalankan eksperimen kedua metode optimisasi

In [ ]:
base_model = build_model(seed=SEED)

model_gd = copy.deepcopy(base_model)
theta_gn_start = flatten_model_params(base_model)

gd_history = train_full_batch_gradient_descent(
    model=model_gd,
    X_train=X_train_tensor,
    y_train=y_train_tensor,
    X_test=X_test_tensor,
    y_test=y_test_tensor,
    lr=0.20,
    epochs=150
)

gn_history, theta_gn_final = train_gauss_newton(
    theta=theta_gn_start,
    X_train=X_train_tensor,
    y_train=y_train_tensor,
    X_test=X_test_tensor,
    y_test=y_test_tensor,
    epochs=15,
    damping=1e-1,
    step_scale=1.0
)

gd_df = pd.DataFrame(gd_history)
gn_df = pd.DataFrame(gn_history)

print("Training selesai.")
display(gd_df.head())
display(gn_df.head())

## 10) Evaluasi akhir

In [ ]:
summary_df = pd.DataFrame({
    "Method": ["Full Batch Gradient Descent", "Gauss-Newton"],
    "Final Loss": [gd_df["loss"].iloc[-1], gn_df["loss"].iloc[-1]],
    "Final Train Accuracy": [gd_df["train_acc"].iloc[-1], gn_df["train_acc"].iloc[-1]],
    "Final Test Accuracy": [gd_df["test_acc"].iloc[-1], gn_df["test_acc"].iloc[-1]],
    "Best Test Accuracy": [gd_df["test_acc"].max(), gn_df["test_acc"].max()],
    "Epochs Run": [len(gd_df), len(gn_df)]
})

summary_df

## 11) Visualisasi loss vs epoch

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(gd_df["epoch"], gd_df["loss"], label="Full Batch GD")
plt.plot(gn_df["epoch"], gn_df["loss"], label="Gauss-Newton")
plt.xlabel("Epoch")
plt.ylabel("Categorical Cross-Entropy Loss")
plt.title("Loss vs Epoch")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 12) Visualisasi accuracy vs epoch

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(gd_df["epoch"], gd_df["train_acc"], label="GD Train Accuracy")
plt.plot(gd_df["epoch"], gd_df["test_acc"], label="GD Test Accuracy")
plt.plot(gn_df["epoch"], gn_df["train_acc"], label="GN Train Accuracy")
plt.plot(gn_df["epoch"], gn_df["test_acc"], label="GN Test Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy vs Epoch")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 13) Analisis jawaban tugas

In [ ]:
def first_epoch_below(values, threshold):
    for i, value in enumerate(values, start=1):
        if value < threshold:
            return i
    return None

gd_epoch_under_01 = first_epoch_below(gd_df["loss"].tolist(), 0.10)
gn_epoch_under_01 = first_epoch_below(gn_df["loss"].tolist(), 0.10)

num_params = theta_gn_start.numel()
num_residuals = X_train_tensor.shape[0] * y_train_tensor.shape[1]
gn_non_monotonic = np.any(np.diff(gn_df["loss"].values) > 0)

print("1) Which method converges faster?")
print(f"   Gauss-Newton mencapai loss < 0.10 pada epoch {gn_epoch_under_01},")
print(f"   sedangkan Full Batch Gradient Descent mencapainya pada epoch {gd_epoch_under_01}.")
print("   Jadi, pada eksperimen ini Gauss-Newton konvergen lebih cepat.")

print("\n2) Is Gauss-Newton stable?")
if gn_non_monotonic:
    print("   Cukup stabil setelah diberi damping, tetapi tidak sepenuhnya monoton.")
    print("   Terlihat ada sedikit kenaikan loss pada beberapa epoch, jadi metode ini cepat namun sensitif.")
else:
    print("   Stabil pada eksperimen ini dan loss turun secara konsisten.")

print("\n3) Compare computational complexity")
print("   - Full Batch GD: satu forward pass + backward pass per epoch.")
print(f"   - Gauss-Newton: perlu Jacobian berukuran ({num_residuals}, {num_params})")
print(f"     lalu menyelesaikan sistem linear ukuran ({num_params}, {num_params}).")
print("   Karena itu, biaya komputasi dan memori Gauss-Newton lebih besar per epoch.")

print("\n4) When is Gauss-Newton preferable?")
print("   Gauss-Newton lebih cocok saat ukuran data/model masih kecil-menengah,")
print("   residual least-squares masih masuk akal, dan kita ingin konvergensi cepat")
print("   walaupun biaya per-epoch lebih mahal.")

## 14) Kesimpulan singkat

- Kedua metode mampu menyelesaikan klasifikasi Iris dengan baik.
- **Gauss-Newton** biasanya mencapai performa tinggi dalam lebih sedikit epoch.
- **Full Batch Gradient Descent** lebih sederhana dan lebih ringan per epoch.
- Untuk tugas skala kecil seperti Iris, Gauss-Newton masih praktis untuk dicoba.